# 02c Manual Tokenizer Generation

## Purpose

This notebook builds the manual glycan tokenizer for the rebuilt workflow.

## Input

- `MyDrive/ProjectRoot2/data/splits/train.txt`

## Outputs

- `MyDrive/ProjectRoot2/tokenizers/manual/<setting_label>/vocab.json`
- Hugging Face tokenizer files saved in the same folder
- `tokenizer_config_summary.json`
- `inspection_preview.csv`

## Notes to myself

This tokenizer is my hand-defined baseline. I'm not learning merges here. I'm fixing the vocabulary directly from the manual parser so I can compare a biologically structured tokenizer against the BPE-based ones later.

## Setup note

Same pattern again.

- code and notebooks stay in GitHub
- tokenizer artifacts stay in Drive
- Colab pulls the repo at the start
- the final cell syncs the notebook back to GitHub

In [1]:
# ==============================================================================
# 0. SET UP THE COLAB ENVIRONMENT
# ==============================================================================
import os
import sys

from google.colab import drive, userdata

drive.mount('/content/drive')

GITHUB_USER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    print('Cloning repository...')
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    print('Repository already exists. Pulling latest changes...')

%cd {REPO_DIR}

!git config --global user.email "hb791-dev@users.noreply.github.com"
!git config --global user.name "hb791-dev"
!git config --global pull.rebase false
!git pull {REPO_URL} main --no-edit -q

if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

print('Colab environment ready.')
print(f'Repo directory: {REPO_DIR}')

Mounted at /content/drive
Cloning repository...
/content/glycan-roberta
Colab environment ready.
Repo directory: /content/glycan-roberta


## Path and setting setup

The manual tokenizer doesn't really have a vocab-size hyperparameter the way BPE does, so I'm using a fixed label here. `v1_train_only` means the vocabulary was built directly from the training split with the manual parser.

In [2]:
# ==============================================================================
# 1. DEFINE THE TRAINING PATHS AND TOKENIZER SETTINGS
# ==============================================================================
PROJECT_ROOT = '/content/drive/MyDrive/ProjectRoot2'
TRAIN_DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'splits', 'train.txt')

SETTING_LABEL = 'v1_train_only'
TOKENIZER_OUT_DIR = os.path.join(PROJECT_ROOT, 'tokenizers', 'manual', SETTING_LABEL)
VOCAB_PATH = os.path.join(TOKENIZER_OUT_DIR, 'vocab.json')

os.makedirs(TOKENIZER_OUT_DIR, exist_ok=True)

print('Training data path:')
print(TRAIN_DATA_PATH)
print('\nTokenizer output directory:')
print(TOKENIZER_OUT_DIR)
print('\nSetting label:')
print(SETTING_LABEL)

if not os.path.exists(TRAIN_DATA_PATH):
    raise FileNotFoundError(f'Training split not found: {TRAIN_DATA_PATH}')

Training data path:
/content/drive/MyDrive/ProjectRoot2/data/splits/train.txt

Tokenizer output directory:
/content/drive/MyDrive/ProjectRoot2/tokenizers/manual/v1_train_only

Setting label:
v1_train_only


## Build the manual vocabulary

This step applies the manual glycan parser to every training sequence, counts the token inventory, and writes a stable `vocab.json`. I'm sorting the learned tokens before assigning IDs so the file is reproducible.

In [ ]:
# ==============================================================================
# 2. BUILD AND SAVE THE MANUAL VOCABULARY
# ==============================================================================
import json
from collections import Counter

from src.tokenizer_utils import split_glycan_string

with open(TRAIN_DATA_PATH, 'r', encoding='utf-8') as file:
    train_sequences = [line.strip() for line in file if line.strip()]

token_counts = Counter()
for sequence in train_sequences:
    token_counts.update(split_glycan_string(sequence))

special_tokens = ['<s>', '<pad>', '</s>', '<unk>', '<mask>']
vocab = {token: index for index, token in enumerate(special_tokens)}

for token in sorted(token_counts.keys()):
    if token not in vocab:
        vocab[token] = len(vocab)

with open(VOCAB_PATH, 'w', encoding='utf-8') as file:
    json.dump(vocab, file, indent=2)

print(f'Manual vocabulary saved to: {VOCAB_PATH}')
print(f'Vocabulary size: {len(vocab)}')
print(f'Unique non-special tokens: {len(vocab) - len(special_tokens)}')

## Compile the Hugging Face tokenizer

Here I'm wrapping the manual vocabulary in a Hugging Face fast tokenizer. The regex pre-tokenizer is important because it makes the saved tokenizer follow the same glycan units as the manual parser.

In [ ]:
# ==============================================================================
# 3. BUILD AND SAVE THE MANUAL HUGGING FACE TOKENIZER
# ==============================================================================
from tokenizers import Regex, Tokenizer, models, pre_tokenizers
from transformers import PreTrainedTokenizerFast

from src.tokenizer_utils import HUGGINGFACE_FAST_PATTERN

with open(VOCAB_PATH, 'r', encoding='utf-8') as file:
    vocab = json.load(file)

word_level_model = models.WordLevel(vocab=vocab, unk_token='<unk>')
backend_tokenizer = Tokenizer(word_level_model)
backend_tokenizer.pre_tokenizer = pre_tokenizers.Split(
    pattern=Regex(HUGGINGFACE_FAST_PATTERN),
    behavior='isolated'
)

hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=backend_tokenizer,
    bos_token='<s>',
    eos_token='</s>',
    unk_token='<unk>',
    pad_token='<pad>',
    mask_token='<mask>'
)

hf_tokenizer.save_pretrained(TOKENIZER_OUT_DIR)

tokenizer_summary = {
    'tokenizer_family': 'manual',
    'setting_label': SETTING_LABEL,
    'train_data_path': TRAIN_DATA_PATH,
    'tokenizer_output_dir': TOKENIZER_OUT_DIR,
    'vocab_size': len(vocab),
    'num_special_tokens': len(special_tokens),
    'num_non_special_tokens': len(vocab) - len(special_tokens),
    'top_10_tokens': token_counts.most_common(10),
    'saved_files': sorted(os.listdir(TOKENIZER_OUT_DIR)),
}

summary_json_path = os.path.join(TOKENIZER_OUT_DIR, 'tokenizer_config_summary.json')
with open(summary_json_path, 'w', encoding='utf-8') as file:
    json.dump(tokenizer_summary, file, indent=2)

print('Manual tokenizer saved.')
print(f'Tokenizer folder: {TOKENIZER_OUT_DIR}')

## Quick sanity check

I don't want deep analysis here. I just want to confirm the saved tokenizer loads and that the first few glycans are being split into the kinds of units I expect.

In [ ]:
# ==============================================================================
# 4. LOAD THE SAVED TOKENIZER AND INSPECT SAMPLE OUTPUT
# ==============================================================================
import pandas as pd

loaded_tokenizer = PreTrainedTokenizerFast.from_pretrained(TOKENIZER_OUT_DIR)

sample_sequences = train_sequences[:3]
inspection_rows = []

for sample_index, sequence in enumerate(sample_sequences, start=1):
    token_ids = loaded_tokenizer.encode(sequence, add_special_tokens=False)
    tokens = loaded_tokenizer.convert_ids_to_tokens(token_ids)

    inspection_rows.append(
        {
            'sample_index': sample_index,
            'sequence': sequence,
            'num_tokens': len(tokens),
            'tokens': ' | '.join(tokens[:30]),
        }
    )

inspection_df = pd.DataFrame(inspection_rows)
display(inspection_df)

print(f'Loaded vocabulary size: {len(loaded_tokenizer)}')
print(f'Mask token: {loaded_tokenizer.mask_token}')
print(f'Pad token: {loaded_tokenizer.pad_token}')

## Save a small inspection table

This gives me a lightweight record of the first manual-tokenizer sanity check without reopening the notebook later.

In [ ]:
# ==============================================================================
# 5. SAVE THE INSPECTION OUTPUT
# ==============================================================================
inspection_path = os.path.join(TOKENIZER_OUT_DIR, 'inspection_preview.csv')
inspection_df.to_csv(inspection_path, index=False)

print(f'Inspection preview saved to: {inspection_path}')

## GitHub sync note

Same idea as the earlier notebooks. The tokenizer files stay in Drive. The notebook itself stays versioned in GitHub.

In [ ]:
# ==============================================================================
# SAVE THE NOTEBOOK BACK TO GITHUB
# ==============================================================================
import json

REPO_NOTEBOOK_PATH = os.path.join(REPO_DIR, 'notebooks/02_tokenizer_generation/02c_manual_gen.ipynb')
NOTEBOOK_FILENAME = os.path.basename(REPO_NOTEBOOK_PATH)
DRIVE_NOTEBOOK_CANDIDATES = [
    f'/content/drive/MyDrive/Colab Notebooks/{NOTEBOOK_FILENAME}',
    f'/content/drive/MyDrive/{NOTEBOOK_FILENAME}',
]

source_notebook_path = None
for candidate in DRIVE_NOTEBOOK_CANDIDATES:
    if os.path.exists(candidate):
        source_notebook_path = candidate
        break

if source_notebook_path is not None:
    !cp "{source_notebook_path}" "{REPO_NOTEBOOK_PATH}"

    # Strip widget metadata if Colab adds it so GitHub rendering stays cleaner.
    try:
        with open(REPO_NOTEBOOK_PATH, 'r', encoding='utf-8') as file:
            notebook_json = json.load(file)

        if 'widgets' in notebook_json.get('metadata', {}):
            del notebook_json['metadata']['widgets']

        with open(REPO_NOTEBOOK_PATH, 'w', encoding='utf-8') as file:
            json.dump(notebook_json, file, indent=1)
    except Exception as exc:
        print(f'Notebook metadata cleanup skipped: {exc}')

    %cd {REPO_DIR}
    !git add notebooks/02_tokenizer_generation/02c_manual_gen.ipynb
    !git commit -m "Update 02c_manual_gen" || echo "No new changes to commit."
    !git pull {REPO_URL} main --no-edit -q
    !git push {REPO_URL} main -q

    print('Notebook synced to GitHub.')
    print(f'Source notebook path: {source_notebook_path}')
else:
    print('No Drive-backed notebook file was found for this session.')
    print('If you opened this notebook directly from GitHub, Colab is editing a browser copy, not a runtime file.')
    print('For GitHub-opened notebooks, use File -> Save a copy in GitHub.')
    print('If you want this cell to auto-sync the notebook, first save or copy the notebook into Drive and then rerun this cell.')
